# 19. Ragas 정량 평가 — Faithfulness · Relevancy · Context P·R
> Day 4 · 22H · 소요 약 50분

## 학습 목표

- RAG 시스템의 4대 평가 메트릭(Faithfulness, Answer Relevancy, Context Precision, Context Recall)을 설명할 수 있다.
- Ragas로 본인 SQL 에이전트를 정량적으로 평가할 수 있다.
- 낮은 점수의 원인을 진단하고 프롬프트를 개선할 수 있다.
- Before/After 비교 차트로 개선 효과를 시각화해 발표 슬라이드에 활용한다.

> **전제 노트북:** 17번 에이전트를 self-contained 로 다시 이식 후 Ragas 입력 형식(question / answer / contexts / ground_truth) 으로 변환합니다.
> **필요 키:** `OPENAI_API_KEY`, `NEON_DSN`. `LANGSMITH_API_KEY` 는 **선택** (있으면 Ragas 내부 판정 호출도 트레이싱됨).

### API 비용 주의

Ragas 의 각 메트릭은 내부적으로 **LLM 판정자(judge)** 를 여러 번 호출합니다. 4 메트릭 × 10 질문 = 약 40~80회 추가 LLM 호출이 발생하며, 판정 LLM을 따로 지정하지 않으면 비용이 크게 오를 수 있습니다. 이 노트북은 **`gpt-4o-mini` 를 판정 LLM으로 명시 주입** 하여 비용을 약 1/10 수준으로 낮춥니다. 처음 실습할 때는 질문 수를 3~5개로 줄여서 파이프라인부터 확인하세요.

### 버전 핀

Ragas 는 0.1 → 0.2 에서 필드 이름이 바뀌었습니다. 본 노트북은 **0.1 계열** (`contexts`, `ground_truth`, `report.to_pandas()`) 기준입니다:

```
pip install "ragas>=0.1.17,<0.2" "datasets>=2.16,<3"
```

In [ ]:
%pip install -q "ragas>=0.1.17,<0.2" "datasets>=2.16,<3" langgraph langchain langchain-openai langsmith sqlalchemy psycopg2-binary sqlparse pandas matplotlib tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os


def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# OpenAI 키와 Neon DSN 은 필수 — Ragas 가 LLM 판정자를 부르고, 우리 에이전트가 DB 를 조회합니다.
_load_secret("OPENAI_API_KEY", required=True)
_load_secret("NEON_DSN", required=True)
# LangSmith 는 선택 — 키가 있으면 Ragas 의 내부 판정 호출까지 트레이싱되어 디버깅이 한결 편해집니다.
_load_secret("LANGSMITH_API_KEY", required=False)
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "sql-agent-day4-ragas")
    os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]

print("Environment ready.")
print(f"  LangSmith tracing = {os.environ.get('LANGCHAIN_TRACING_V2', 'off')}")

## 1. 4대 메트릭 요약

| 메트릭 | 묻는 것 | 낮을 때 원인 | 튜닝 방향 |
|---|---|---|---|
| **Faithfulness** | 답변이 컨텍스트에 근거하는가 (= 지어내지 않았는가) | 할루시네이션, LLM 사전지식 유출 | 프롬프트에 "컨텍스트 밖 정보 금지" 명시, temperature=0 |
| **Answer Relevancy** | 답변이 질문에 직접 답하는가 | 관련 없는 답, 일반론 | "질문에 직접 답하세요" 규칙, 불필요한 배경 생략 |
| **Context Precision** | 검색된 컨텍스트 중 실제 기여한 비율 | 불필요한 문서가 많음 | Re-rank, 메타데이터 필터, TopK↓ |
| **Context Recall** | 정답에 필요한 정보를 컨텍스트가 담는가 | 필요한 문서 누락 | TopK↑, 하이브리드 검색, 스키마 COMMENT 보강 |

### SQL 에이전트에서의 해석 주의

Context Precision/Recall은 **여러 문서를 검색하는 순수 RAG** 를 전제로 설계되었습니다. 우리 에이전트는 `contexts` 에 "SQL + 결과" 한 덩어리만 들어가므로:

- **Context Precision**: 거의 항상 1에 가깝게 나옵니다 (문서가 1개뿐).
- **Context Recall**: SQL 결과가 ground_truth 의 사실들을 포함하는지를 봅니다 → 사실상 **"SQL 생성 품질"의 간접 지표**.

발표에서는 이 한계를 언급하면 "평가의 한계까지 인지" 한 것으로 높이 평가됩니다.

## 2. 17번 에이전트 재이식 (self-contained)

NB18 과 동일하게 17번 에이전트를 이식합니다. 로직 / 상태 / 노드 / 재시도 분기 모두 17번과 1:1 일치.

In [ ]:
# ============================================================
# 2. 17번 에이전트 재이식 — 의존성 import + DB 엔진
# ============================================================
# TODO: `sqlalchemy.create_engine`으로 Neon 엔진을 만들고(읽기 전용 옵션 시도 후 실패 시 일반 모드로 폴백), `langgraph`/`langchain_openai`/`langchain_core` 모듈을 import 하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 스키마 수집 함수
# ============================================================
# TODO: `sqlalchemy.inspect`로 컬럼·FK를 가져와 `CREATE TABLE` 문자열을 만들고, `information_schema.columns` + `col_description`으로 COMMENT를 붙이는 `collect_schema(engine, tables)`를 작성하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# AgentState + 가드레일
# ============================================================
# TODO: `TypedDict`로 `question/sql/sql_result/error/answer/attempts` 필드를 가진 `AgentState`를 정의하고, 정규식으로 DDL/DML(`DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE`) 키워드를 차단하는 `sanitize_sql`을 만드세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 노드 함수들
# ============================================================
# TODO: Day 3과 동일하게 `generate_sql` (프롬프트→LLM→파싱), `run_sql` (`sanitize_sql`+`pd.read_sql`), `validate`, `answer` (결과 요약 프롬프트), `should_retry` (조건부 분기) 5개 노드를 작성하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# 그래프 조립
# ============================================================
# TODO: `StateGraph(AgentState)`로 4개 노드를 추가하고 `generate_sql → run_sql → validate`로 엣지를 연결, `validate`에 `should_retry` 조건부 엣지를 걸어 `compile()` 하세요.
# 여기에 구현하세요.


## 3. 평가 데이터 수집 — 10개 질문 실행

Ragas 는 4필드 (`question`, `answer`, `contexts`, `ground_truth`) 의 DataFrame을 요구합니다. 먼저 에이전트를 10번 돌려 답변을 수집한 뒤, SQL + SQL 결과를 `contexts` 로 엮고 사전에 준비된 ground_truth 사전으로 정답을 붙입니다.

In [ ]:
# ============================================================
# 3. 평가 데이터 수집 — 10개 질문 실행
# ============================================================
# TODO: 10개 질문 리스트를 순회하며 `agent.invoke(_initial_state(q))`를 호출하고, question/sql/result_md/answer/error/retries를 `results` 리스트에 수집하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# Ragas 입력 형식 변환 (ground_truths_dict + eval_data)
# ============================================================
# TODO: Ragas 0.1 입력 4개 필드(`question`, `answer`, `contexts`, `ground_truth`)를 가진 dict를 만들고, 질문→정답 매핑 `ground_truths_dict`를 정의한 뒤 `results`를 순회하며 `contexts`에는 `f"SQL: {sql}\n결과: {sql_result}"` 한 덩어리를 리스트로 넣으세요.
# 여기에 구현하세요.


## 4. Ragas 평가 실행

판정 LLM을 `gpt-4o-mini` 로 주입해 비용을 낮춥니다. 평가 수행 시간은 질문 수 × 4 메트릭 × 수 초 정도 = **대략 2~5분**. 진행 로그가 멈춘 듯 보여도 기다리세요.

In [ ]:
# ============================================================
# 4. Ragas 평가 실행
# ============================================================
# TODO: `datasets.Dataset.from_dict(eval_data)`로 변환한 뒤 `ragas.evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb)`를 호출하세요. 비용 절감을 위해 판정자는 `LangchainLLMWrapper(ChatOpenAI("gpt-4o-mini"))` + `LangchainEmbeddingsWrapper(OpenAIEmbeddings("text-embedding-3-small"))`로 주입하고, 결과는 `report.to_pandas()` 후 메트릭 컬럼 `mean()`으로 평균을 구하세요.
# 여기에 구현하세요.


## 5. 질문별 상세 점수

In [ ]:
# ============================================================
# 5. 질문별 상세 점수
# ============================================================
# TODO: `report.to_pandas()`로 DataFrame을 만든 뒤 `question` + 4개 메트릭 컬럼을 골라 `to_string(index=False)`로 질문별 점수를 출력하세요.
# 여기에 구현하세요.


## 6. 시각화 — 4-패널 막대 차트

빨간 점선(0.7) 아래의 질문이 "개선이 필요한" 대상입니다.

In [ ]:
# ============================================================
# 6. 시각화 — 4-패널 막대 차트
# ============================================================
# TODO: `plt.subplots(2, 2)`로 4개 패널을 만들고, 메트릭마다 `ax.barh`로 질문별 점수를 그린 뒤 `axvline(x=0.7)`로 목표선을 표시하세요.
# 여기에 구현하세요.


### 레이더 차트 — 전체 요약

4개 축이 균형 있게 바깥쪽으로 뻗어 있으면 좋은 에이전트. 한 축이 안쪽으로 들어가 있다면 그 메트릭이 약점입니다.

In [ ]:
# ============================================================
# 레이더 차트 — 전체 요약
# ============================================================
# TODO: `subplot_kw=dict(polar=True)`로 polar axes를 만들고, 4개 메트릭 평균값을 `np.linspace(0, 2π, n, endpoint=False)` 각도에 매핑한 뒤 첫 점을 끝에 한 번 더 추가하여 원형을 닫고 `ax.plot` + `ax.fill`로 그리세요.
# 여기에 구현하세요.


## 7. 낮은 점수 원인 진단

각 질문·메트릭을 스캔해 0.7 미만인 항목에 "의심 원인 + 처방" 을 자동 주석으로 답니다. 이 진단이 곧 튜닝 방향의 출발점입니다.

In [ ]:
# ============================================================
# 7. 낮은 점수 원인 진단
# ============================================================
# TODO: `df_eval`을 행 단위로 순회하며 메트릭 값이 0.7 미만인 항목을 모은 뒤, 메트릭 이름별로 진단(할루시네이션/답변 관련성 등) + 처방(프롬프트 보강/Re-ranking/COMMENT 추가 등)을 출력하세요.
# 여기에 구현하세요.


## 8. v2 튜닝 — 답변 프롬프트 개선

대표적인 개선 중 하나로 `generate_answer` 노드의 프롬프트를 **더 엄격하게** 바꿔 Faithfulness / Answer Relevancy 를 함께 노려봅니다. 아래 v2 프롬프트는:

- "**결과에 있는 정보만 사용하라**" 명시 (할루시네이션 억제 → Faithfulness↑)
- "**질문에 직접 답하라**" 지시 (Answer Relevancy↑)
- 숫자 천단위 구분 / 추측 금지 규칙 유지

> 튜닝은 **한 번에 한 변수만** 바꾸는 것이 원칙. 동시에 여러 개를 바꾸면 어떤 변경이 어떤 메트릭을 움직였는지 추적 불가능.

In [ ]:
# ============================================================
# 8. v2 튜닝 — answer_v2 개선 프롬프트 + 그래프 재구성
# ============================================================
# TODO: 기존 `answer` 노드를 복제한 `answer_v2`를 작성하되, 프롬프트에 "결과에 있는 정보만 사용", "결과에 없는 정보 금지", "질문에 직접 답변" 같은 명시적 규칙을 추가해 할루시네이션을 억제하고, 동일한 `StateGraph`를 다시 짜되 `answer` 자리에 `answer_v2`를 등록해 `agent_v2`로 컴파일하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# v2 에이전트 재실행 + eval_data_v2 구성
# ============================================================
# TODO: `LANGSMITH_PROJECT`(또는 `LANGCHAIN_PROJECT`)를 v2용으로 바꾼 뒤 동일한 10개 질문을 다시 `agent_v2.invoke`하여 `results_v2`에 모으고, 같은 형식으로 `eval_data_v2`를 만드세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# v2 Ragas 재평가
# ============================================================
# TODO: v1과 동일한 형식의 `eval_data_v2`를 사용해 같은 메트릭/판정자 설정으로 `evaluate()`를 다시 호출한 뒤 `to_pandas()` 평균으로 `metric_means_v2`를 구하세요.
# 여기에 구현하세요.


## 9. Before / After 비교

숫자 표와 나란히 놓인 막대 차트 두 가지 포맷으로 뽑습니다. **막대 차트는 발표 슬라이드 3번째 장의 핵심 시각 자료**로 사용하세요.

In [ ]:
# ============================================================
# 9. Before / After 비교 — 표 출력
# ============================================================
# TODO: 4개 메트릭에 대해 `metric_means`(v1)와 `metric_means_v2`(v2) 평균을 나란히 출력하고, 차이(`v2 - v1`)와 방향(UP/DOWN/SAME)을 표 형태로 표시하세요.
# 여기에 구현하세요.


In [ ]:
# ============================================================
# Before / After 비교 bar 차트 (발표 슬라이드용)
# ============================================================
# TODO: `np.arange(len(metrics))`를 x축으로 두고 `width=0.35` 간격으로 v1/v2 두 벌의 `ax.bar`를 그려 그룹 막대 차트를 만든 뒤, `axhline(y=0.7)`로 목표선을 그리고 PNG로 저장하세요.
# 여기에 구현하세요.


## 실습 과제

1. **본인 에이전트의 10개 질문에 대해 Ragas 평가를 실행**하세요.
2. **가장 낮은 점수의 질문을 찾아 원인을 진단**하세요:
    - 검색 실패인가요? (Context Recall 낮음)
    - 할루시네이션인가요? (Faithfulness 낮음)
    - 질문과 관련 없는 답변인가요? (Answer Relevancy 낮음)
3. **하나 이상의 개선을 적용**하고 Before/After 비교를 기록하세요 (발표에 사용).
4. (도전) 프롬프트 외에 다른 개선도 시도해보세요:
    - 스키마 COMMENT 추가
    - 검색 범위 조정
    - generate_sql 프롬프트 개선

**생각해보기**

- 4대 메트릭 중 **가장 개선하기 쉬운 것**은 무엇일까요? 가장 어려운 것은?
- Faithfulness와 Answer Relevancy는 **프롬프트 튜닝**으로 개선할 수 있습니다. Context Precision과 Context Recall은 어떻게 개선할까요?
- Ragas 점수가 1.0이면 완벽한 에이전트일까요? 점수가 높아도 실제 사용에서 문제가 될 수 있는 상황은?
- 만약 10개 질문 중 2개만 점수가 낮다면, 전체 평균을 보는 것보다 **질문별 점수**를 보는 것이 더 유용한 이유는 무엇일까요?


In [ ]:
# ============================================================
# 실습 과제 — 본인 프로젝트에 Ragas 평가 적용
# ============================================================
# TODO: `questions`/`ground_truths_dict`를 본인 도메인 질문·정답으로 교체한 뒤 본인 에이전트를 실행해 `results`를 수집하고, 같은 4개 메트릭으로 `evaluate()`를 호출해 가장 낮은 메트릭 1개에 한정하여 v2 튜닝을 적용한 후 Before/After 비교 차트를 PNG로 저장하세요.
# 여기에 구현하세요.


## 다음 노트북에서는...

**더 이상 노트북은 없습니다.** 23H는 최종 튜닝 & 리허설 시간(개인 작업 + 강사 1:1 피드백), 24H는 최종 발표 & 수료식입니다. 준비물은:

- **에이전트 v1** — 17H·18H·19H 에서 다듬은 최종 버전.
- **LangSmith Trace URL** — 18H 에서 기록된 본인 프로젝트 링크.
- **Ragas 리포트** — `ragas_report.png` + `ragas_before_after.png` + 숫자 표.
- **발표 슬라이드 3장** — 문제 정의 / 아키텍처 / 결과 & 회고.

최종 발표 5~7분. 라이브 데모 2~4개 질문 + Ragas Before/After 차트 공유가 핵심입니다. 실패 사례와 디버깅 과정을 당당히 설명하세요 — 그것이 점수의 절반입니다.